In [1]:
import duckdb
import pandas as pd
import numpy as np

# =====================================================================
# 📦 靶场数据物理投料 (Raw Data Input)
# =====================================================================
# 模拟用户画像标签，由于风控打标签有时间差，导致每个人绑定的标签数量严重不对等！
user_profiles = pd.DataFrame({
    'user_id': [9001, 9002, 9003, 9004],
    'username': ['张三', '李四', '王五', '赵六'],
    'tags': [
        ['高频转账', '设备异常', '疑似黑产'], # 9001: 长度为 3，内鬼在第三格
        ['疑似黑产', '白条逾期'],           # 9002: 长度为 2，内鬼在第一格
        ['优质白领'],                     # 9003: 长度为 1，安全用户
        None                              # 9004: 长度为 0 (缺失值)，安全用户
    ]
})

print("🔥 原始高危用户标签池投料成功：")
print(user_profiles)

🔥 原始高危用户标签池投料成功：
   user_id username                tags
0     9001       张三  [高频转账, 设备异常, 疑似黑产]
1     9002       李四        [疑似黑产, 白条逾期]
2     9003       王五              [优质白领]
3     9004       赵六                None


**只要用户的标签抽屉里，包含 `疑似黑产` 的标签，无论它藏在第几个格子里，统统揪出来！**

### 🎯 轨道一：PostgreSQL / DuckDB 轨道

- **审查官刚性要求**：
    
    1. 必须使用 **`ANY` 散射算子**，物理轰炸整个抽屉，把藏在不同位置的 `9001` 和 `9002` 精准截获。
        
    2. 顺便利用 `tags[1]` 抠出每一个人的**首要特征（第一格标签）**，命名为 `primary_tag`。
        
    3. 输出字段：`user_id`, `primary_tag`。

In [ ]:
sql_query = """
SELECT  user_id,
        tags[1] AS primary_tag
FROM    user_profiles
WHERE   '疑似黑产' = ANY(tags) 
ORDER BY user_id;
"""
df_sql = duckdb.query(sql_query).df()
print(df_sql)

   user_id primary_tag
0     9001        高频转账
1     9002        疑似黑产


### 🎯 轨道二：Pandas 工业级链式轨道

在接下来的 Cell 里，用 **Lambda 链式匿名函数流** 迎击。

- **审查官刚性要求**：
    
    1. 必须在提取 `primary_tag` 之前，用 **`fillna()` 黑魔法** 把 `None/NaN` 的坑物理填成空列表 `[]`，彻底阻断 Python 的越界暴毙地雷。
        
    2. 利用安全算子抠出第一格特征（注意 Python 的 0 基索引！）。
        
    3. 利用布尔过滤或高效筛选，揪出包含 `疑似黑产` 的行。
        
    4. 最终用 `pd.testing.assert_frame_equal(..., check_dtype=False)` 进行拉闸交会对账！

In [6]:
# =====================================================================
# 🐼 轨道二：PANDAS 工业级完全体（布尔遮罩收网流）—— 彻底免去 query 隐式解析坑
# =====================================================================

# 1. 物理清洗 + 压榨特征
df_processed = user_profiles.assign(
    tags = lambda x: x['tags'].apply(lambda d: d if isinstance(d, list) else []),
    primary_tag = lambda x: x['tags'].str[0]
)

# 2. 🌟 既然拦截点 3 的 apply 已经算出了最精确的内鬼盾牌，直接用它做刚性行拦截
is_fraud = df_processed['tags'].apply(lambda x: '疑似黑产' in x)

# 3. 拦截过滤、字段裁剪、像素级清洗一气呵成
df_final = (
    df_processed[is_fraud] # 🛡️ 盾牌直接卡位，True 的放行，False 的物理剔除！
    [['user_id', 'primary_tag']]
    .sort_values(by='user_id')
    .reset_index(drop=True)
)

print("🏆【内鬼终究伏法！】Pandas 轨道最终清洗结果：")
print(df_final)

# =====================================================================
# 🚨 跨语言特征合围对账
# =====================================================================
pd.testing.assert_frame_equal(
    df_sql.reset_index(drop=True), 
    df_final.reset_index(drop=True),
    check_dtype=False
)
print("\n🎉 🎉 🎉【跨语言对账天衣无缝！】SQL 轨道的 @> 与 Pandas 轨道像素级全绿对齐！")

🏆【内鬼终究伏法！】Pandas 轨道最终清洗结果：
   user_id primary_tag
0     9001        高频转账
1     9002        疑似黑产

🎉 🎉 🎉【跨语言对账天衣无缝！】SQL 轨道的 @> 与 Pandas 轨道像素级全绿对齐！
